## Language Modeling
### It is the task of predicting the next word (or character) in a sequence based on the context of previous word

In [2]:
!pip install nltk

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from nltk.tokenize import word_tokenize
import nltk
from collections import Counter


In [4]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""


In [5]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
#Tokenize
tokens=word_tokenize(document.lower())

In [7]:
vocab={'<UNK>':0}
for token in Counter(tokens).keys():
  if token not in vocab :
    vocab[token]=len(vocab)
len(vocab)

289

In [8]:
# extract sentence from data
input_sentence = document.split('\n')

In [9]:
input_sentence

['About the Program',
 'What is the course fee for  Data Science Mentorship Program (DSMP 2023)',
 'The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.',
 'What is the total duration of the course?',
 'The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)',
 'What is the syllabus of the mentorship program?',
 'We will be covering the following modules:',
 'Python Fundamentals',
 'Python libraries for Data Science',
 'Data Analysis',
 'SQL for Data Science',
 'Maths for Machine Learning',
 'ML Algorithms',
 'Practical ML',
 'MLOPs',
 'Case studies',
 'You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390',
 'Will Deep Learning and NLP be a part of this program?',
 'No, NLP and Deep Learning both are not a part of this program’s curriculum.',
 'What if I miss a live session? Will I get a recording 

In [10]:
def text_to_indices(sentence, vocab):
    indices=[]
    for token in sentence:
      if token not in vocab:
        indices.append(vocab['<UNK>'])
      else:
        indices.append(vocab[token])
    return indices

In [11]:
input_numerical_sentence=[]
for sentence in input_sentence:
    input_numerical_sentence.append(text_to_indices(word_tokenize(sentence.lower()),vocab))

In [13]:
len(input_numerical_sentence)

78

In [14]:
training_sequence=[]
for sentence in input_numerical_sentence:
  for i in range (1,len(sentence)):
    training_sequence.append(sentence[:i+1])


In [15]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [4, 5], [4, 5, 2], [4, 5, 2, 6]]

In [16]:
len_list=[]
for sequence in training_sequence:
  len_list.append(len(sequence))
max(len_list) # value for pre padding

62

In [17]:
padded_sequence=[]
for sequence in training_sequence:
    padded_sequence.append([0]*(max(len_list)-len(sequence))+sequence)


In [18]:
padded_sequence=torch.tensor(padded_sequence, dtype=torch.long)

In [19]:
 padded_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   0,   4,   5],
        ...,
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287],
        [  0,   0,   0,  ..., 286, 287, 288]])

In [20]:
X=padded_sequence[:,:-1]
y=padded_sequence[:,-1]

In [21]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   0,   0,   4],
        ...,
        [  0,   0,   0,  ...,   0, 285, 176],
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287]])

In [22]:
class CustomDataset(Dataset):
      def __init__(self,X,y):
        self.X=X
        self.y=y
      def __len__(self):
        return self.X.shape[0]
      def __getitem__(self, index):
         return self.X[index] , self.y[index]

In [23]:
dataset=CustomDataset(X,y)

In [24]:
len(dataset)

942

In [25]:
data_loader=DataLoader(dataset,batch_size=32,shuffle=True)

In [37]:
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100)
    self.lstm=nn.LSTM(100,150,batch_first=True)
    self.fc=nn.Linear(150,vocab_size)
  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_hidden_state,(final_hidden_state,final_cell_state)=self.lstm(embedded)
    output=self.fc(final_hidden_state.squeeze(0))
    return output


In [38]:
model=LSTMModel(len(vocab))


In [41]:
epochs=50
learning_rate=0.001
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [42]:
# training loop
for epoch in range(epochs):
  total_loss=0
  for batch_x , batch_y in data_loader:
    optimizer.zero_grad()
    output=model(batch_x)
    loss=criterion(output,batch_y)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch : {epoch+1}, Loss: {total_loss:4f}")


Epoch : 1, Loss: 61.744419
Epoch : 2, Loss: 52.431390
Epoch : 3, Loss: 46.968403
Epoch : 4, Loss: 41.212269
Epoch : 5, Loss: 37.011787
Epoch : 6, Loss: 32.480142
Epoch : 7, Loss: 28.796182
Epoch : 8, Loss: 25.350603
Epoch : 9, Loss: 22.299737
Epoch : 10, Loss: 20.268188
Epoch : 11, Loss: 17.989283
Epoch : 12, Loss: 16.090705
Epoch : 13, Loss: 14.510503
Epoch : 14, Loss: 13.072017
Epoch : 15, Loss: 11.883385
Epoch : 16, Loss: 10.956919
Epoch : 17, Loss: 9.978601
Epoch : 18, Loss: 9.381187
Epoch : 19, Loss: 8.706265
Epoch : 20, Loss: 8.228389
Epoch : 21, Loss: 7.638209
Epoch : 22, Loss: 7.263074
Epoch : 23, Loss: 6.845223
Epoch : 24, Loss: 6.545409
Epoch : 25, Loss: 6.290501
Epoch : 26, Loss: 6.057634
Epoch : 27, Loss: 5.924013
Epoch : 28, Loss: 5.692536
Epoch : 29, Loss: 5.584319
Epoch : 30, Loss: 5.233963
Epoch : 31, Loss: 5.180106
Epoch : 32, Loss: 5.074838
Epoch : 33, Loss: 4.879407
Epoch : 34, Loss: 4.814580
Epoch : 35, Loss: 4.613807
Epoch : 36, Loss: 4.623210
Epoch : 37, Loss: 4.4

In [43]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]



In [44]:
prediction(model, vocab, "The course follows a monthly")

'The course follows a monthly subscription'

In [45]:
import time

num_tokens = 10
input_text = "hi how are"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


hi how are to
hi how are to make
hi how are to make all
hi how are to make all the
hi how are to make all the past
hi how are to make all the past lectures
hi how are to make all the past lectures ?
hi how are to make all the past lectures ? ?
hi how are to make all the past lectures ? ? will
hi how are to make all the past lectures ? ? will i
